<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/Neural-Recommendation-Personalization-Engine/blob/main/01_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import os
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

rm: cannot remove '/content/drive/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.Trash-0': Directory not empty
rm: cannot remove '/content/drive/.Encrypted/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/.Encrypted/MyDrive': Operation canceled
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q huggingface_hub
from huggingface_hub import login,HfApi
login()

KeyboardInterrupt: 

In [ ]:
# !pip install -q pandas==2.2.3
# !pip -q install -U huggingface_hub pandas pyarrow
# !pip install datasets faiss-cpu polars -q

In [ ]:
!pip install -U datasets huggingface_hub

In [ ]:
url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/benchmark/5core/rating_only/Electronics.csv"
reviews = pd.read_csv(url)

print("Shape:", reviews.shape)
reviews.head()

Shape: (15473536, 4)


,user_id,parent_asin,rating,timestamp
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000


In [ ]:
print('Shape:',reviews.shape,'\n\n')
print('Columns:')
print(reviews.columns.tolist(),'\n')
print('Data Types:',reviews.dtypes,'\n')
print('Missing Values:',reviews.isna().sum())
print('\nDuplicate Rows:',reviews.duplicated().sum())
print('\n Rating Distribution',reviews['rating'].value_counts().sort_index())

print("\nUnique Users:", reviews["user_id"].nunique())
print("Unique Products:", reviews["parent_asin"].nunique())

display(reviews.head())

Shape: (15473536, 4) 


Columns:
['user_id', 'parent_asin', 'rating', 'timestamp'] 

Data Types: user_id         object
parent_asin     object
rating         float64
timestamp        int64
dtype: object 

Missing Values: user_id        0
parent_asin    0
rating         0
timestamp      0
dtype: int64

Duplicate Rows: 0

 Rating Distribution rating
1.0     1287788
2.0      713558
3.0     1074820
4.0     2190347
5.0    10207023
Name: count, dtype: int64

Unique Users: 1641026
Unique Products: 368228


,user_id,parent_asin,rating,timestamp
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B0047T79VS,3.0,1344406083000
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01HHURN3W,3.0,1408995743000
2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B00L0YLRUW,1.0,1439226089000
3,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,5.0,1456772365000
4,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,5.0,1456772571000


In [3]:
base_url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw_meta_Electronics/"

for i in range(10):
    shard_url = f"{base_url}full-0000{i}-of-00010.parquet"

    globals()[f"meta{i+1}"] = pd.read_parquet(shard_url)

    print(f"meta{i+1} loaded: {globals()[f'meta{i+1}'].shape}")

meta1 loaded: (161002, 16)
meta2 loaded: (161002, 16)
meta3 loaded: (161001, 16)
meta4 loaded: (161001, 16)
meta5 loaded: (161001, 16)
meta6 loaded: (161001, 16)
meta7 loaded: (161001, 16)
meta8 loaded: (161001, 16)
meta9 loaded: (161001, 16)
meta10 loaded: (161001, 16)


In [7]:
meta10.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Computers,"Samsung Galaxy Tab A 10.1 Case, Kickstand Armo...",3.6,4,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",X-Tablet,"[Electronics, Computers & Accessories, Tablet ...","{""Package Dimensions"": ""10.8 x 6.6 x 1 inches""...",B078Z6L24Z,None,None,None
1,Computers,LeiJue Keyboard Case for iPad 10.2(7th/8th/9th...,4.0,5,[Compatibility: Design for iPad 9th Generation...,[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",LeiJue,"[Electronics, Computers & Accessories, Tablet ...","{""Package Dimensions"": ""11.14 x 8.31 x 1.34 in...",B09TVKK86T,None,None,None
2,Computers,Laptop Keyboard for Gateway M360-3401701 M460A...,3.0,1,[],[],None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Gateway,"[Electronics, Computers & Accessories, Compute...","{""Manufacturer"": ""Gateway"", ""Date First Availa...",B000X56KV0,None,None,None
3,Portable Audio & Accessories,iCloth Lens and Screen Cleaner Pro-Grade Indiv...,5.0,5,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",iCloth,"[Electronics, Television & Video, Accessories,...","{""Brand Name"": ""iCloth"", ""Item Weight"": ""0.64 ...",B01K3NNEDS,None,None,None
4,Computers,Boskin for AMZ F i r e 7 case 2019 2017 Releas...,4.5,74,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Boskin,"[Electronics, Computers & Accessories, Tablet ...","{""Brand"": ""Boskin"", ""Compatible Devices"": ""Tab...",B08B1V1T2S,None,None,None


In [7]:
shard.shape

(161001, 16)

In [9]:
check_columns = [
    "main_category", "title", "average_rating", "rating_number",
    "price", "store", "details", "parent_asin",
    "bought_together", "subtitle", "author"
]

for i in range(1, 11):
    df = globals()[f"meta{i}"]

    print(f"\n{'='*50}")
    print(f"METADATA {i}")
    print(f"{'='*50}")

    print("Shape:", df.shape)
    print("\nColumns:", df.columns.tolist())
    print("\nData Types:\n", df.dtypes)
    print("\nMissing Values:\n", df.isna().sum())

    duplicates = df[check_columns].duplicated().sum()
    print("\nDuplicate Rows:", duplicates)


METADATA 1
Shape: (161002, 16)

Columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

Data Types:
 main_category       object
title               object
average_rating     float64
rating_number        int64
features            object
description         object
price               object
images              object
videos              object
store               object
categories          object
details             object
parent_asin         object
bought_together     object
subtitle            object
author              object
dtype: object

Missing Values:
 main_category           0
title                   0
average_rating          0
rating_number           0
features                0
description             0
price                   0
images                  0
videos                  0
store                   0
categories  

In [ ]:
meta1 = meta1.drop(columns=['bought_together','subtitle','author',"images", "videos"])
print(meta1.isna().sum())


main_category     1642
title                0
average_rating       0
rating_number        0
features             0
description          0
price                0
store              890
categories           0
details              0
parent_asin          0
dtype: int64


In [8]:
for i in range(1, 11):
    df = globals()[f"meta{i}"]
    df["main_category"] = df["main_category"].fillna(
        df["main_category"].mode()[0])
    df["store"] = df["store"].fillna(
        df["store"].mode()[0])

    globals()[f"meta{i}"] = df

In [ ]:
user_interaction = reviews.groupby('user_id').size()
item_interaction = reviews.groupby('parent_asin').size()

print("User Interaction Distribution:")
print(user_interaction.describe())

print("\nItem Interaction Distribution:")
print(item_interaction.describe())

User Interaction Distribution:
count    1.641026e+06
mean     9.429184e+00
std      8.752549e+00
min      5.000000e+00
25%      5.000000e+00
50%      7.000000e+00
75%      1.000000e+01
max      9.380000e+02
dtype: float64

Item Interaction Distribution:
count    368228.000000
mean         42.021617
std         231.315077
min           5.000000
25%           7.000000
50%          12.000000
75%          27.000000
max       44948.000000
dtype: float64


In [ ]:
print("Minimum timestamp:",reviews['timestamp'].min())
print("Maximum timestamp:",reviews['timestamp'].max())

Minimum timestamp: 929311804000
Maximum timestamp: 1694509255248


In [ ]:
num_users = reviews['user_id'].nunique()
num_items = reviews['parent_asin'].nunique()
num_interaction = len(reviews)

sparsity = 1 - (num_interaction / (num_users * num_items))

print("Users:", num_users)
print("Items:", num_items)
print("Interactions:", num_interaction)
print("Sparsity:", sparsity)

Users: 1641026
Items: 368228
Interactions: 15473536
Sparsity: 0.9999743930827167


In [ ]:
# Create NeuraRec folders in actual Google Drive
os.makedirs("/content/drive/MyDrive/NeuraRec/data/raw",exist_ok=True)

# Save datasets
reviews.to_parquet("/content/drive/MyDrive/NeuraRec/data/raw/reviews.parquet",index=False)
meta.to_parquet("/content/drive/MyDrive/NeuraRec/data/raw/metadata.parquet",index=False)
print("Saved!")
print(os.listdir("/content/drive/MyDrive/NeuraRec/data/raw"))

Saved!
['reviews.parquet', 'metadata.parquet']


In [ ]:
api = HfApi()

api.upload_file(path_or_fileobj="/content/drive/MyDrive/NeuraRec/data/raw/reviews.parquet",
               path_in_repo="raw/reviews.parquet",
                repo_id="Satyamkarn100/NeuraRec-Electronics",
                repo_type="dataset")
api.upload_file(path_or_fileobj="/content/drive/MyDrive/NeuraRec/data/raw/metadata.parquet",
                       path_in_repo="raw/metadata.parquet",
                       repo_id="Satyamkarn100/NeuraRec-Electronics",
                       repo_type='dataset')


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../data/raw/reviews.parquet:   8%|7         | 23.9MB /  313MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...data/raw/metadata.parquet:  16%|#6        | 24.0MB /  148MB            

CommitInfo(commit_url='https://huggingface.co/datasets/Satyamkarn100/NeuraRec-Electronics/commit/6c0cd1a3662ccbd43d134e4967d2872dd6411055', commit_message='Upload raw/metadata.parquet with huggingface_hub', commit_description='', oid='6c0cd1a3662ccbd43d134e4967d2872dd6411055', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Satyamkarn100/NeuraRec-Electronics', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Satyamkarn100/NeuraRec-Electronics'), pr_revision=None, pr_num=None)

In [ ]:
print(os.path.isdir("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive")[:20])